# Instalamos dependencias

In [3]:
pip install --upgrade pip

  Using cached pip-25.3-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-25.3-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2
Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install plotly jinja2 weasyprint kaleido pandas numpy

  Using cached pandas-2.3.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-2.3.3-cp313-cp313-macosx_11_0_arm64.whl (10.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 20.1 MB/s  0:00:00eta 0:00:01
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pandas]2m3/4 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install nbformat --upgrade

Note: you may need to restart the kernel to use updated packages.


# Cargamos el dataset
### Seleccionamos la universidad de la que generaremos el informe

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/Users/danielgomez/Desktop/TFG/data/clean-data/La_Laguna/Tabla.csv", sep=';')

# 2. Limpieza de seguridad (por si quedaron ??? o ..)
def limpiar_valor(v):
    if v in ["??", "???", "..", "None"] or pd.isna(v):
        return np.nan
    return float(str(v).replace(',', '.'))

df['Valor'] = df['Valor'].apply(limpiar_valor)

# Generamos grafico

In [15]:
# Colores institucionales
carrera = "Informática"
colores_genero = {"Hombres": "#1f77b4", "Mujeres": "#e377c2", "Ambos Sexos": "#7f7f7f"}
df_filtrado = df[df['Carrera'] == carrera]

## Faceted Bar Chart

In [32]:
import plotly.express as px

fig = px.bar(
        df_filtrado, 
        x="Anio", y="Valor", color="Genero",
        facet_col="Tasa", # Divide en 3 columnas por cada tasa
        barmode="group",
        title=f"Panel de Control Académico: {carrera}",
        labels={"Valor": "Porcentaje (%)", "Anio": "Curso Académico"},
        color_discrete_map=colores_genero,
        text_auto='.1f'
    )
fig.update_xaxes(autorange="reversed")
fig.write_image(f"../output/graph/G1_Resumen_{carrera}.png")
fig.show()

## Evolución de la Brecha Neta (Line Chart)

In [33]:
import plotly.express as px

df_pivot = df_filtrado.pivot_table(index=["Anio", "Tasa"], columns="Genero", values="Valor").reset_index()
df_pivot['Brecha'] = df_pivot['Mujeres'] - df_pivot['Hombres']

fig = px.line(
    df_pivot, x="Anio", y="Brecha", color="Tasa",
    title=f"Evolución de la Brecha de Género (M - H): {carrera}",
    markers=True
)
fig.add_hline(y=0, line_dash="dash", line_color="black", annotation_text="Igualdad")
fig.update_xaxes(autorange="reversed")
fig.write_image(f"../output/graph/G2_Brecha_{carrera}.png")
fig.show()

## Velocímetro de Rendimiento

In [34]:
import plotly.express as px
import plotly.graph_objects as go

# Obtenemos el último año y la tasa de rendimiento
ultimo_anio = df_filtrado['Anio'].max()
valor_actual = df_filtrado[(df_filtrado['Anio'] == ultimo_anio) & 
                            (df_filtrado['Tasa'] == "Rendimiento") & 
                            (df_filtrado['Genero'] == "Ambos Sexos")]['Valor'].values[0]

fig = go.Figure(go.Indicator(
    mode = "gauge+number",
    value = valor_actual,
    title = {'text': f"Rendimiento Actual ({ultimo_anio})"},
    gauge = {'axis': {'range': [0, 100]},
                'bar': {'color': "#1f77b4"},
                'steps': [
                    {'range': [0, 50], 'color': "#ff9999"},
                    {'range': [50, 80], 'color': "#ffff99"},
                    {'range': [80, 100], 'color': "#99ff99"}]}
))
fig.write_image(f"../output/graph/G3_Velocimetro_{carrera}.png")
fig.show()

## Comparativa contra la Media (Bullet Chart)

In [35]:
import plotly.express as px
import plotly.graph_objects as go

ultimo_anio = df['Anio'].max()
# Valor de la carrera
val_carrera = df[(df['Carrera'] == carrera) & (df['Anio'] == ultimo_anio) & (df['Tasa'] == "Rendimiento") & (df['Genero'] == "Ambos Sexos")]['Valor'].values[0]
# Media global
val_media = df[(df['Carrera'] == "Todos los ámbitos") & (df['Anio'] == ultimo_anio) & (df['Tasa'] == "Rendimiento") & (df['Genero'] == "Ambos Sexos")]['Valor'].values[0]

fig = go.Figure(go.Indicator(
    mode = "number+gauge+delta",
    value = val_carrera,
    delta = {'reference': val_media},
    domain = {'x': [0, 1], 'y': [0, 1]},
    title = {'text': f"Rendimiento {carrera} vs Media Univ."},
    gauge = {
        'shape': "bullet",
        'axis': {'range': [None, 100]},
        'threshold': {
            'line': {'color': "red", 'width': 2},
            'thickness': 0.75,
            'value': val_media}, # La línea roja es la media de la universidad
        'bar': {'color': "#1f77b4"}
    }
))
fig.write_image(f"../output/graph/G4_CompMedia_{carrera}.png")
fig.show()

## Tabla de Resumen Estadístico

In [23]:
import plotly.express as px
import plotly.graph_objects as go

# Resumen de los últimos 3 años
df_tabla = df_filtrado[df_filtrado['Genero'] != "Ambos Sexos"].tail(9) 

fig = go.Figure(data=[go.Table(
    header=dict(values=['Año', 'Tasa', 'Género', 'Valor (%)'],
                fill_color='paleturquoise', align='left'),
    cells=dict(values=[df_tabla.Anio, df_tabla.Tasa, df_tabla.Genero, df_tabla.Valor],
                fill_color='lavender', align='left'))
])
fig.show()

# Escribir informe

In [36]:
pip install fpdf2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [fpdf2]
Note: you may need to restart the kernel to use updated packages.


In [56]:
import pandas as pd
from jinja2 import Template
from datetime import datetime
import os
from pathlib import Path

# 1. Configuración de datos
carrera_actual = "Informática"
fecha_hoy = datetime.now().strftime("%d/%m/%Y")

def ruta_a_uri(ruta_relativa):
    # Convierte ruta relativa a absoluta y luego a formato URI (file:///...)
    return Path(ruta_relativa).resolve().as_uri()

datos_informe = {
    "carrera": carrera_actual,
    "fecha": fecha_hoy,
    "tasa_tipo": "Rendimiento / Éxito / Evaluación",
    "tendencia_g1": "estable con ligero crecimiento en el último bienio",
    "path_g1": ruta_a_uri(f"../output/graph/G1_Resumen_{carrera_actual}.png"),
    "path_g2": ruta_a_uri(f"../output/graph/G2_Brecha_{carrera_actual}.png"),
    "path_g3": ruta_a_uri(f"../output/graph/G3_Velocimetro_{carrera_actual}.png"),
    "path_g4": ruta_a_uri(f"../output/graph/G4_CompMedia_{carrera_actual}.png"),
}

# 3. Leer la plantilla HTML
with open("../templates/informe_prueba.html", "r", encoding="utf-8") as f:
    plantilla_html = f.read()

# 4. Renderizar (inyectar datos)
template = Template(plantilla_html)
html_final = template.render(datos_informe)

# 5. Guardar el HTML resultante (para revisar o convertir)
with open(f"../output/informe_html/informe_{carrera_actual}.html", "w", encoding="utf-8") as f:
    f.write(html_final)

print(f"HTML generado para {carrera_actual}. Ahora puedes convertirlo a PDF.")

HTML generado para Informática. Ahora puedes convertirlo a PDF.


In [39]:
pip install weasyprint

Note: you may need to restart the kernel to use updated packages.


In [57]:
from weasyprint import HTML
HTML(string=html_final).write_pdf(f"../output/informe_pdf/Informe_{carrera_actual}.pdf")